# Data Leakage:
* Data leakage is when information from outside the training data — specifically, information the model shouldn't have access to when making a real prediction — accidentally sneaks into the training process. The result: our model looks great during evaluation (great MAE, high accuracy) but performs much worse in the real world, because it learned to "cheat" using information it won't actually have at prediction time.
* It's one of the most dangerous ML bugs because it doesn't throw an error — it silently makes our model look better than it actually is. This connects directly to bugs we have already hit this session, even though we didn't call them "leakage" at the time.
* Note: `Data leakage = our evaluation number is lying to us. The model looks better than it actually is, because somewhere, some piece of information it shouldn't have had access to yet — either a feature that only exists after the fact, or a peek at the "hidden" test data — snuck in during training.`
### There are two main types of leakage:
#### Target leakage:
`A feature only exists because the answer already happened — it secretly contains the answer in disguise.
Example: predicting rain using "did people carry umbrellas" — umbrellas come after rain starts, so this feature won't exist yet when you actually need a prediction.
Test: would this feature's value be known before the outcome happens? If not → target leakage.`
#### Train-Test Contamination:
`Validation/test data accidentally influences how training data is prepared — so the "unseen" test isn't really unseen anymore.
Example: fitting an imputer (mean/mode) on train + valid combined, instead of train only — validation data quietly shapes its own preprocessing.
Fix: always fit() only on train, transform() (never re-fit) on valid/test.`

In this example, we will learn one way to detect and remove target leakage.

We will use a dataset about credit card applications. The end result is that information about each credit card application is stored in a DataFrame X. We'll use it to predict which applications were accepted in a Series y

In [19]:
import pandas as pd
data=pd.read_csv('AER_credit_card_data.csv',
                true_values=['yes'],false_values=['no']) #This tells pandas: "wherever you see the exact text 'yes' in the file,
#treat it as boolean True. Wherever you see 'no', treat it as boolean False."

data.dropna(subset=['card'],axis=0,inplace=True)
y=data.card
X=data.drop(['card'],axis=1)
X.head()

,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,0,37.66667,4.5200,0.033270,124.983300,True,False,3,54,1,12
1,0,33.25000,2.4200,0.005217,9.854167,False,False,3,34,1,13
2,0,33.66667,4.5000,0.004156,15.000000,True,False,4,58,1,5
3,0,30.50000,2.5400,0.065214,137.869200,False,False,0,25,1,7
4,0,32.16667,9.7867,0.067051,546.503300,True,False,2,64,1,5


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

model=RandomForestClassifier(n_estimators=100,random_state=0)
my_pipeline=Pipeline(steps=[
    ('model',model)
])

validation_scores=cross_val_score(
    my_pipeline,
    X,y,
    cv=5,
    scoring='accuracy'
)

print(f"Cross-Validation Accuracy: {validation_scores.mean()*100}%")

Cross-Validation Accuracy: 98.02915082382764%
